In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [ ]:

import json
import pandas as pd
from pathlib import Path

# 1. Read raw NASA POWER JSON from Bronze
# Replace the workspace name only if yours differs.
bronze_path = Path(
    "/lakehouse/lh_bronze/Files/nasa_power/raw/nasa_power_uae_2024.json"
)

print("Bronze file exists:", bronze_path.exists())

if not bronze_path.exists():
    raise FileNotFoundError(
        f"Bronze file not accessible: {bronze_path}"
    )

# 2. Load the original NASA POWER response
with open(bronze_path, "r", encoding="utf-8") as f:
    raw_weather = json.load(f)

# 3. Extract daily weather parameters
weather_parameters = raw_weather["properties"]["parameter"]

df_silver_weather = pd.DataFrame(weather_parameters)

# 4. Convert date from YYYYMMDD to datetime
df_silver_weather.index.name = "date"
df_silver_weather = df_silver_weather.reset_index()

df_silver_weather["date"] = pd.to_datetime(
    df_silver_weather["date"],
    format="%Y%m%d"
)

# 5. Standardize column names and units
df_silver_weather = df_silver_weather.rename(
    columns={
        "ALLSKY_SFC_SW_DWN": "solar_irradiation_kwh_m2_day",
        "T2M": "air_temperature_c",
        "WS2M": "wind_speed_m_s",
        "RH2M": "relative_humidity_pct"
    }
)

# 6. Add source and location metadata
df_silver_weather["location_name"] = "Dubai_UAE_Scenario"
df_silver_weather["latitude"] = 24.9857
df_silver_weather["longitude"] = 55.1761
df_silver_weather["source_system"] = "NASA_POWER"

# 7. Validate data
print("Rows:", len(df_silver_weather))
print("Columns:", df_silver_weather.columns.tolist())
print("Duplicate dates:", df_silver_weather["date"].duplicated().sum())

print("\nMissing values:")
print(df_silver_weather.isnull().sum())

display(df_silver_weather.head(10))


StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can't be run because you've hit spark overall capacity compute limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.